# 技能1 · Day 3 上机：企业知识图谱 + GraphRAG

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **networkx** 构建营销知识图谱（产品-品牌-品类-客户-评论-活动-渠道），执行图查询
2. 用 **numpy** 从零实现 TransE KGE（h+r≈t），理解知识图谱嵌入的训练过程
3. 实现并对比**传统RAG**（TF-IDF 向量检索）与**GraphRAG**（知识图谱多跳检索）在营销多跳问答上的效果差异
4. 理解 GraphRAG 的核心创新：实体关系抽取 + 多跳推理 + 社区摘要

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：networkx（图计算）+ numpy（KGE）+ scikit-learn（传统RAG基线）。
营销映射：构建企业营销知识图谱，用 GraphRAG 回答"买X的用户还买什么"等多跳问题。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ networkx/numpy/scikit-learn 是纯 Python 库，无需外部服务。
> langchain-experimental 的 LLMGraphTransformer 需 LLM API Key（上机中作为参考展示，不强制运行）。

In [ ]:
# !pip install networkx numpy scikit-learn -q
# 可选（LLMGraphTransformer 需 API Key）：
# !pip install langchain-experimental langchain-openai -q
# export OPENAI_API_KEY=sk-...

## 1. 数据集背景与营销映射

**构建对象**：企业营销知识图谱，覆盖7类实体和8类关系：

| 实体类型 | 示例 | 关系类型 | 示例 |
|---------|------|---------|------|
| Product | 智能跑步手表ProMax(1299元) | PURCHASED | 客户001 -> 智能跑步手表ProMax |
| Brand | TechFit、SoundWave | MANUFACTURED_BY | 产品 -> 品牌 |
| Category | 智能穿戴设备、音频设备 | BELONGS_TO | 产品 -> 品类 |
| Customer | 客户001-004 | COMPETES_WITH | 产品 -> 产品 |
| Campaign | 2026春季跑步节 | COMPLEMENTARY_TO | 产品 -> 产品 |
| Channel | 小红书、抖音、微信公众号 | REVIEWED | 客户 -> 产品（带评分） |

**营销映射**：在真实项目中，这些数据来自 CRM/电商/客服系统。本上机用预置的真实场景数据。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import networkx as nx
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("networkx:", nx.__version__)
print("numpy:", np.__version__)
print("导入完成")

## TODO 1：用 networkx 构建营销知识图谱

**核心任务**：创建一个 MultiDiGraph，添加产品/品牌/品类/客户/活动/渠道节点，以及 PURCHASED/MANUFACTURED_BY/BELONGS_TO/COMPETES_WITH/COMPLEMENTARY_TO/REVIEWED/PROMOTES/PROMOTED_THROUGH/PARTICIPATED_IN 等关系边。

**为什么用 MultiDiGraph**：营销关系是多类型有向边（同一对节点间可能有多种关系），MultiDiGraph 原生支持。

In [ ]:
# TODO 1：用 networkx 构建营销知识图谱
# 提示：用 MultiDiGraph 创建图，添加产品/品牌/品类/客户/活动/渠道节点
#   再添加 PURCHASED/MANUFACTURED_BY/BELONGS_TO/COMPETES_WITH 等关系边
#   G.add_node(name, type=..., price=...)
#   G.add_edge(src, dst, relation=...)
# 要求：构建包含4个产品、2个品牌、4个客户的知识图谱

# ===== 你的代码 =====
G = nx.MultiDiGraph()

# 添加产品节点（4个产品，带price/category属性）
# TODO: 添加 智能跑步手表ProMax(1299元)、智能健康手环Lite(299元)、无线降噪耳机Pro(899元)、运动蓝牙耳机Mini(199元)

# 添加品牌节点
# TODO: 添加 TechFit、SoundWave

# 添加品类、客户、活动、渠道节点
# TODO: 添加 智能穿戴设备/音频设备 品类，客户001-004，2026春季跑步节，小红书/抖音/微信公众号

# 添加关系边
# TODO: MANUFACTURED_BY（产品->品牌）、BELONGS_TO（产品->品类）
# TODO: COMPETES_WITH（产品->产品）、COMPLEMENTARY_TO（产品->产品）
# TODO: PURCHASED（客户->产品，带timestamp/quantity属性）
# TODO: REVIEWED（客户->产品，带rating/text属性）
# TODO: PROMOTES（活动->产品）、PROMOTED_THROUGH（活动->渠道）、PARTICIPATED_IN（客户->活动）
# ====================

print(f"知识图谱: {G.number_of_nodes()} 个节点, {G.number_of_edges()} 条边")
print(f"实体类型: {set(n.get('type','') for _, n in G.nodes(data=True))}")

## 2. KGE 理论：TransE / RotatE / ComplEx

知识图谱嵌入（KGE）把实体和关系映射到低维向量空间，使得三元组 (h, r, t) 可在向量空间中被表示和预测。

**TransE 核心思想**：h + r ≈ t（头实体向量 + 关系向量 ≈ 尾实体向量）

```
得分函数：f_r(h, t) = -||h + r - t||
训练损失（margin-based ranking）：L = Σ max(0, γ + f(h,t) - f(h',t'))
    γ = margin（间隔），(h',r,t') = 负采样的错误三元组
```

| 方法 | 核心 | 适用关系 | 局限 |
|:----:|------|---------|------|
| TransE | h+r≈t（平移） | 一对一 | 无法处理一对多 |
| RotatE | h∘r≈t（复数旋转） | 一对多/多对多 | 实现复杂 |
| ComplEx | Re(h̄·diag(r)·t) | 对称+非对称 | 需复数运算 |

本 TODO 用 numpy 从零实现 TransE，理解训练过程的每一步。

## TODO 2：TransE KGE 实现（numpy）

**核心任务**：从知识图谱中提取三元组，用 numpy 实现 TransE 的嵌入训练。
- 提取 (h, r, t) 三元组
- 初始化实体/关系嵌入矩阵
- 训练循环：负采样 -> 计算 margin loss -> 梯度更新
- 验证：已知三元组的 h+r 与 t 的距离应减小

In [ ]:
# TODO 2：TransE KGE 实现（numpy）
# 提示：从图中提取三元组 (h, r, t)，初始化嵌入，训练 TransE
#   核心公式：h + r ≈ t
#   得分函数：f_r(h, t) = -||h + r - t||
#   损失：L = Σ max(0, γ + f(h,t) - f(h',t'))（margin-based ranking）
#   负采样：随机替换尾实体生成错误三元组 (h, r, t')
# 要求：训练200轮，打印loss，验证已知三元组的 h+r≈t

# ===== 你的代码 =====
# 1. 提取三元组
triples = []  # TODO: 从 G.edges(data=True) 提取 (head, tail, relation)
entities = set()  # TODO: 收集所有实体（节点）
relations = set()  # TODO: 收集所有关系类型

# 2. 创建ID映射
entity2id = {}  # TODO: {entity_name: id}
relation2id = {}  # TODO: {relation_name: id}

# 3. 初始化嵌入（numpy）
dim = 50
np.random.seed(42)
entity_emb = None  # TODO: np.random.randn(num_entities, dim) / dim
relation_emb = None  # TODO: np.random.randn(num_relations, dim) / dim

# 4. 训练 TransE
margin = 1.0
lr = 0.01
num_epochs = 200
triples_ids = []  # TODO: 将三元组转换为ID格式
# TODO: 训练循环（每50轮打印loss）
#   - 负采样：随机替换尾实体
#   - 计算 pos_diff = h+r-t, neg_diff = h+r-t_neg
#   - margin loss = max(0, margin + ||pos_diff||^2 - ||neg_diff||^2)
#   - 梯度更新：h,r 用 grad=2*(pos_diff-neg_diff)；t 用 -2*pos_diff；t_neg 用 2*neg_diff
# ====================

# 验证 h + r ≈ t
print("\n验证 h + r ≈ t（已知三元组）：")
for h, r, t in triples_ids[:5]:
    # TODO: 计算并打印 ||entity_emb[h] + relation_emb[r] - entity_emb[t]||
    pass

## 3. 知识图谱查询：图算法的真正价值

networkx 提供丰富的图算法，这是手写字典无法做到的：

| 查询类型 | 算法 | 营销应用 |
|---------|------|---------|
| 最短路径 | `nx.shortest_path` | 两个产品间的关联路径 |
| 邻居节点 | `G.neighbors` | 产品的直接关联实体 |
| 社区发现 | `nx.community.louvain_communities` | 客户/产品聚类 |
| 中心性 | `nx.degree_centrality` | 识别核心产品/关键客户 |

## TODO 3：知识图谱查询（最短路径/邻居/社区发现/中心性）

**核心任务**：用 networkx 图算法查询知识图谱，发现产品间的关联路径、社区结构和关键节点。

In [ ]:
# TODO 3：知识图谱查询（最短路径/邻居/社区发现/中心性）
# 提示：用 networkx 的图算法查询知识图谱
#   nx.shortest_path(G.to_undirected(), source, target) -- 最短路径（无向图，忽略边方向）
#   G.neighbors(node) -- 邻居节点
#   nx.community.louvain_communities(G.to_undirected()) -- 社区发现
#   nx.degree_centrality(G) / nx.betweenness_centrality(G) -- 中心性
# 要求：执行4种图查询，打印结果

# ===== 你的代码 =====
# 1. 最短路径：智能跑步手表ProMax 到 无线降噪耳机Pro
shortest_path = None  # TODO: nx.shortest_path(G.to_undirected(), source=..., target=...)

# 2. 邻居：智能跑步手表ProMax 的所有邻居
neighbors = None  # TODO: list(G.neighbors(...))

# 3. 社区发现：用 Louvain 算法发现社区
communities = None  # TODO: nx.community.louvain_communities(G.to_undirected())

# 4. 中心性分析：度中心性 + 介数中心性
degree_cent = None  # TODO: nx.degree_centrality(G)
betweenness_cent = None  # TODO: nx.betweenness_centrality(G)
# ====================

print(f"1. 最短路径: {' -> '.join(shortest_path)}")
print(f"2. 邻居: {neighbors}")
print(f"3. 社区数量: {len(communities)}")
for i, comm in enumerate(communities):
    print(f"   社区{i}: {comm}")
print(f"4. 中心性 Top-3:")
for node in sorted(degree_cent, key=degree_cent.get, reverse=True)[:3]:
    print(f"   {node}: 度={degree_cent[node]:.3f}, 介数={betweenness_cent[node]:.3f}")

## 4. 传统RAG：向量检索的局限

传统RAG的工作流程：`用户提问 -> TF-IDF向量化 -> 余弦相似度检索 -> Top-K文档 -> 拼入Prompt -> LLM生成`

**传统RAG的局限**：
1. **无法做多跳推理**："买X的用户还买什么"需要 Product -> Customer -> Product 两跳，文本检索做不到
2. **缺乏关系理解**：只看语义相似度，不理解文档间的结构关系
3. **全局问题困难**：无法综合多个文档的信息

本 TODO 用 scikit-learn 的 TfidfVectorizer 实现传统RAG基线。

## TODO 4：传统RAG实现（TF-IDF 向量检索）

**核心任务**：用 scikit-learn 实现 TF-IDF 向量检索，作为对比基线。用多跳问题测试，观察其局限。

In [ ]:
# TODO 4：传统RAG实现（TF-IDF 向量检索）
# 提示：用 sklearn 的 TfidfVectorizer 将文档向量化
#   注意：中文文本无空格分词，默认tokenizer会把整句当一个token
#   解决：用 analyzer='char', ngram_range=(2,3) 做字符级n-gram（无需额外库）
#   用 cosine_similarity 计算查询与文档的相似度
#   取 Top-K 文档作为检索结果
# 要求：实现传统RAG，用多跳问题测试，观察其局限

# ===== 你的代码 =====
# 营销文档（作为知识库）
documents = [
    # TODO: 定义5个营销文档（产品文档/活动文档）
    # 文档1: 智能跑步手表ProMax的产品文档
    # 文档2: 智能健康手环Lite的产品文档
    # 文档3: 无线降噪耳机Pro的产品文档
    # 文档4: 运动蓝牙耳机Mini的产品文档
    # 文档5: 2026春季跑步节活动文档
]

# TF-IDF 向量化
vectorizer = None  # TODO: TfidfVectorizer(analyzer='char', ngram_range=(2, 3))
doc_vectors = None  # TODO: vectorizer.fit_transform(documents)

# 查询并检索
query = "购买智能跑步手表ProMax的用户还买了什么？"
query_vec = None  # TODO: vectorizer.transform([query])
similarities = None  # TODO: cosine_similarity(query_vec, doc_vectors).flatten()

# Top-K 检索
top_k = 3
top_indices = None  # TODO: similarities.argsort()[-top_k:][::-1]
# ====================

print(f"查询: {query}")
print(f"\n传统RAG检索结果（Top-{top_k}）：")
for idx in top_indices:
    print(f"  [相似度={similarities[idx]:.4f}] {documents[idx][:60]}...")
print("\n局限：传统RAG只能基于文本相似度检索，无法做多跳关系推理。")

## 5. GraphRAG：知识图谱多跳检索

GraphRAG（微软2024, arXiv 2404.16130）的核心创新：用知识图谱的边做多跳推理检索。

**GraphRAG vs 传统RAG**：
```
传统RAG：  问题 -> 文本相似度 -> Top-K文档 -> 答案
GraphRAG： 问题 -> 实体定位 -> 沿边多跳推理 -> 精确答案
```

**三种多跳查询模式**：
1. **co_purchase**：Product <- PURCHASED <- Customer -> PURCHASED -> Product（买X的用户还买什么）
2. **brand_products**：Brand <- MANUFACTURED_BY <- Product（品牌旗下产品）
3. **competitors**：Product -> COMPETES_WITH -> Product（竞品查询）

## TODO 5：GraphRAG实现（知识图谱多跳检索）

**核心任务**：实现 `graphrag_query` 函数，用知识图谱的多跳检索回答营销问题。
- co_purchase：找到购买指定产品的客户，再找这些客户购买的其他产品
- brand_products：找到指定品牌的所有产品及属性
- competitors：找到指定产品的竞品及属性

**参考**：LLMGraphTransformer 可用 LLM 从文本自动抽取实体关系构建 KG（需 API Key，本上机作为参考展示）。

In [ ]:
# TODO 5：GraphRAG实现（知识图谱多跳检索）
# 提示：GraphRAG 沿知识图谱的边做多跳推理检索
#   查询1 co_purchase：Product <- PURCHASED <- Customer -> PURCHASED -> Product
#   查询2 brand_products：Brand <- MANUFACTURED_BY <- Product
#   查询3 competitors：Product -> COMPETES_WITH -> Product
# 要求：实现 graphrag_query 函数，回答3个多跳问题

# ===== 你的代码 =====
def graphrag_query(graph, query_type, entity):
    """GraphRAG多跳检索：从实体出发，沿关系边做多跳检索"""
    results = []

    if query_type == "co_purchase":
        # 查询：购买entity的用户还买了什么？
        # 步骤1：找到购买entity的客户（Customer -> PURCHASED -> entity）
        customers = None  # TODO: 遍历 graph.edges(data=True) 找 PURCHASED 指向 entity 的客户
        # 步骤2：找到这些客户购买的其他产品
        for customer in (customers or []):
            pass  # TODO: 遍历该客户的 PURCHASED 边，找到其他产品，加入 results

    elif query_type == "brand_products":
        # 查询：品牌entity旗下有哪些产品？
        products = None  # TODO: 遍历边找 MANUFACTURED_BY 指向 entity 的产品
        for prod in (products or []):
            pass  # TODO: 获取产品属性（price/category），加入 results

    elif query_type == "competitors":
        # 查询：和entity竞争的产品有哪些？
        competitors = None  # TODO: 遍历边找 entity 的 COMPETES_WITH 关系
        for comp in (competitors or []):
            pass  # TODO: 获取竞品属性，加入 results

    return results
# ====================

# 测试3个多跳查询
print("GraphRAG 查询1: 购买智能跑步手表ProMax的用户还买了什么？")
results = graphrag_query(G, "co_purchase", "智能跑步手表ProMax")
for r in results:
    print(f"  {r}")

print("\nGraphRAG 查询2: TechFit品牌旗下有哪些产品？")
results = graphrag_query(G, "brand_products", "TechFit")
for r in results:
    print(f"  {r}")

print("\nGraphRAG 查询3: 和无线降噪耳机Pro竞争的产品有哪些？")
results = graphrag_query(G, "competitors", "无线降噪耳机Pro")
for r in results:
    print(f"  {r}")

## 6. GraphRAG vs 传统RAG 效果对比

完成 TODO 4-5 后，我们有了两种检索方法。TODO 6 用4个多跳问题对比两者的召回率：

| 问题类型 | 传统RAG | GraphRAG |
|---------|---------|---------|
| 买X的用户还买什么？ | 不能（需两跳推理） | 能（沿PURCHASED边） |
| 品牌旗下有哪些产品？ | 部分（可能漏产品） | 能（沿MANUFACTURED_BY边） |
| 和X竞争的产品有哪些？ | 不能（需关系推理） | 能（沿COMPETES_WITH边） |

**召回率** = 能回答的问题数 / 总问题数

## TODO 6：GraphRAG vs 传统RAG 效果对比

**核心任务**：定义4个多跳问题，分别用传统RAG和GraphRAG回答，计算召回率并打印对比表格。

In [ ]:
# TODO 6：GraphRAG vs 传统RAG 效果对比
# 提示：定义4个多跳问题，分别用传统RAG和GraphRAG回答，对比召回率
#   传统RAG：用TODO4的TF-IDF检索，检查Top-1文档是否直接包含答案
#   GraphRAG：用TODO5的graphrag_query，检查返回结果是否非空
#   召回率 = 能回答的问题数 / 总问题数
# 要求：打印对比表格，计算两种方法的召回率

# ===== 你的代码 =====
test_queries = [
    {"question": "购买智能跑步手表ProMax的用户还买了什么？", "type": "co_purchase", "entity": "智能跑步手表ProMax"},
    {"question": "TechFit品牌旗下有哪些产品？", "type": "brand_products", "entity": "TechFit"},
    {"question": "和无线降噪耳机Pro竞争的产品有哪些？", "type": "competitors", "entity": "无线降噪耳机Pro"},
    {"question": "购买智能健康手环Lite的用户还买了什么？", "type": "co_purchase", "entity": "智能健康手环Lite"},
]

traditional_correct = 0
graphrag_correct = 0
total = len(test_queries)

print(f"{'问题':<35} {'传统RAG':<12} {'GraphRAG':<12}")
print("-" * 65)

for q in test_queries:
    # 传统RAG检索
    qv = None  # TODO: vectorizer.transform([q['question']])
    sims = None  # TODO: cosine_similarity(qv, doc_vectors).flatten()
    # 传统RAG能否直接回答多跳问题？通常不能（需要关系推理）
    trad_can_answer = False

    # GraphRAG多跳检索
    gr_results = None  # TODO: graphrag_query(G, q['type'], q['entity'])
    graphrag_can_answer = False  # TODO: len(gr_results) > 0

    if trad_can_answer:
        traditional_correct += 1
    if graphrag_can_answer:
        graphrag_correct += 1

    trad_str = "能" if trad_can_answer else "不能"
    gr_str = "能" if graphrag_can_answer else "不能"
    print(f"{q['question'][:33]:<35} {trad_str:<12} {gr_str:<12}")
# ====================

print(f"\n传统RAG召回率: {traditional_correct/total:.1%}")
print(f"GraphRAG召回率: {graphrag_correct/total:.1%}")

## 7. 反思与前沿

### 反思问题
1. GraphRAG 在哪个营销场景下显著优于传统RAG？为什么？（提示：多跳关系推理 vs 语义相似度匹配）
2. TransE 的 h+r≈t 在一对一关系上效果好，但"客户-购买-多个产品"是一对多关系--TransE会有什么问题？RotatE如何解决？
3. 如果知识图谱中有错误的三元组（如错误的竞品关系），GraphRAG 的检索结果会怎样？如何保证图谱质量？
4. GraphRAG 的构建成本（LLM抽取实体关系）高于传统RAG（只需向量化）--什么场景下值得这个成本？

### 2026 前沿：GraphRAG + KGE + 图检索增强
- **GraphRAG**（微软2024, arXiv 2404.16130）：用 LLM 自动构建KG + Leiden社区检测 + 社区摘要，支持 Global/Local/DRIFT 三种搜索
- **KGE**：TransE/RotatE/ComplEx 将实体关系映射到向量空间，支持链接预测
- **LangGraph**：LangChain 生态的图式 Agent 框架，支持构建基于图的 RAG 管道，实现 ReAct 风格多步推理
- **RAGAS**：RAG 评估框架，用 LLM-as-a-judge 评估 GraphRAG vs 传统RAG 的检索质量

**注意**：GraphRAG 增强了可解释性和关系推理能力，但对应因果阶梯 L1（关联分析），不能替代真实业务验证（L2 A/B测试）。

参考 [arXiv 2404.16130](https://arxiv.org/abs/2404.16130)（GraphRAG）+ [networkx](https://networkx.org/)。